# VarChat — Qwen2.5-7B QLoRA Fine-tune (Google Colab / ücretsiz T4)

Bu notebook:
1. `egitim_verisi.jsonl` (148 kaynaklı Türkçe örnek) ile **Qwen2.5-7B-Instruct**'ı **QLoRA (4-bit)** ile eğitir,
2. eğitilmiş modeli **PubMed retrieval** ile birleştirip **uçtan uca** test eder,
3. LoRA'yı **GGUF**'a çevirir → yerelde **Ollama**'ya taşınır.

**ÖNEMLİ:** Menü → *Runtime → Change runtime type → **T4 GPU*** seçili olmalı.

> Not: Ollama yerel bir sunucu olduğu için Colab'da modeli doğrudan `transformers` ile çalıştırıyoruz.
> Retrieval (c01) mantığı bu notebook'a gömülüdür; yereldeki `c04_varchat_ollama.py` ile aynı system prompt kullanılır.

## 0) GPU kontrolü

In [ ]:
!nvidia-smi

## 1) Kütüphaneler

In [ ]:
!pip install -q -U "transformers>=4.44,<4.46" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" "datasets>=2.20"
print("kuruldu")

## 2) Eğitim verisini getir

**Seçenek A — repoyu klonla** (tercih edilen; `fine_tune/` GitHub'a push edilmişse):
aşağıdaki klonlama satırlarının yorumunu kaldır.

**Seçenek B — tek dosya yükle** (varsayılan): çalıştır, açılan pencereden `egitim_verisi.jsonl` seç.

In [ ]:
# --- Seçenek A: tüm proje (retrieval kodu + veri birlikte gelir) ---
# !git clone https://github.com/atu-ce/varchat.git
# %cd varchat
# DATA_PATH = "fine_tune/egitim_verisi.jsonl"

# --- Seçenek B: sadece eğitim verisini yükle ---
from google.colab import files
up = files.upload()                      # egitim_verisi.jsonl'i seç
DATA_PATH = "egitim_verisi.jsonl"

import os
assert os.path.exists(DATA_PATH), f"Bulunamadi: {DATA_PATH}"
print("veri:", DATA_PATH)

## 3) (İsteğe bağlı) Google Drive'a bağlan
Eğitilmiş adapter'ı kalıcı saklamak için (Colab oturumu kapanınca dosyalar silinir).
Gerekmiyorsa bu hücreyi atla.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# KAYIT_DIZINI = '/content/drive/MyDrive/varchat-lora'   # Drive'a kaydet
KAYIT_DIZINI = 'qwen7b-varchat-lora'                    # yerel (oturumluk)
print("kayit dizini:", KAYIT_DIZINI)

## 4) Modeli 4-bit (QLoRA) yükle

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 bf16 desteklemez -> fp16
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 5) Veriyi hazırla (chat şablonu + token uzunluğu)

Her örnek `system + user + assistant` mesajlarından oluşuyor. Qwen'in **chat şablonunu**
uygulayıp tek metne çeviriyoruz. KAYNAKLAR (5 abstract) uzun olduğu için token uzunluğuna bakıp
`MAX_LEN`'i ona göre seçiyoruz.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=DATA_PATH, split="train")

def bicimle(ornek):
    metin = tok.apply_chat_template(ornek["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": metin}

ds = ds.map(bicimle, remove_columns=ds.column_names)

# Token uzunluk dagilimi (MAX_LEN secimi icin)
uzunluklar = [len(tok(x["text"]).input_ids) for x in ds]
uzunluklar.sort()
print("ornek sayisi:", len(uzunluklar))
print("min / medyan / maks token:", uzunluklar[0], uzunluklar[len(uzunluklar)//2], uzunluklar[-1])

MAX_LEN = 4096   # cogu ornegi kapsar; OOM olursa 3072 veya 2048'e dusur
print("MAX_LEN:", MAX_LEN)

In [ ]:
def tokenle(ornek):
    return tok(ornek["text"], truncation=True, max_length=MAX_LEN)

ds_tok = ds.map(tokenle, remove_columns=["text"])

from transformers import DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tok, mlm=False)   # nedensel LM (mlm=False)
print("hazir:", ds_tok)

## 6) Eğitim (QLoRA)

148 örnek × 3 epoch, T4'te yaklaşık **20-45 dk**. Basitlik için tüm metin üzerinde eğitiyoruz
(sadece cevaba maskeleme yapmıyoruz — PoC için yeterli).

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir=KAYIT_DIZINI,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,       # etkin batch = 8
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    fp16=True, bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=ds_tok, data_collator=collator)
trainer.train()

In [ ]:
# Adapter'i kaydet (yalnizca LoRA agirliklari ~ birkac yuz MB)
trainer.save_model(KAYIT_DIZINI)
tok.save_pretrained(KAYIT_DIZINI)
print("kaydedildi ->", KAYIT_DIZINI)

## 7) Hızlı test — eğitim verisindeki bir varyant (biçim/dil kontrolü)

In [ ]:
model.config.use_cache = True
model.eval()

def uret(system, user, max_new_tokens=600):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Egitim verisinden ilk ornegin system+user'ini al, modeli calistir
import json
with open(DATA_PATH, encoding="utf-8") as f:
    ilk = json.loads(f.readline())
m = {x["role"]: x["content"] for x in ilk["messages"]}
print("SORU:", m["user"], "\n")
print(uret(m["system"], m["user"]))

## 8) Uçtan uca test — YENİ varyant (retrieval + üretim)

Eğitimde **olmayan** bir varyant girip PubMed'den canlı makale çekiyoruz, `c04`'teki
system prompt'u kurup eğitilmiş modele özet ürettiriyoruz. Gerçek genelleme testi budur.

> Retrieval fonksiyonları `c01_makale_getir.py` / `c03`'ten birebir kopyadır (Colab'da tek dosya bağımsız çalışsın diye).

In [ ]:
import requests, xml.etree.ElementTree as ET

ESEARCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
EFETCH  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
EMAIL, TOOL = "yazilimbirimi@karacasutekstil.com.tr", "varchat-tez-prototip"

def makale_ara(term, adet=5):
    p = {"db":"pubmed","term":term,"retmax":adet,"retmode":"json","sort":"relevance","email":EMAIL,"tool":TOOL}
    r = requests.get(ESEARCH, params=p, timeout=30); r.raise_for_status()
    s = r.json()["esearchresult"]
    return s["idlist"], int(s.get("count", 0))

def _butun_metin(el):
    return "" if el is None else "".join(el.itertext()).strip()

def makale_detaylari_al(idler):
    if not idler: return []
    p = {"db":"pubmed","id":",".join(idler),"rettype":"abstract","retmode":"xml","email":EMAIL,"tool":TOOL}
    r = requests.get(EFETCH, params=p, timeout=30); r.raise_for_status()
    kok = ET.fromstring(r.text); out = []
    for a in kok.findall(".//PubmedArticle"):
        baslik = _butun_metin(a.find(".//ArticleTitle")) or "(baslik yok)"
        ozet = " ".join(x for x in (_butun_metin(e) for e in a.findall(".//AbstractText")) if x) or "(ozet yok)"
        pid = a.find(".//PMID")
        out.append({"pmid": pid.text if pid is not None else "?", "baslik": baslik, "ozet": ozet})
    return out

def baglam_metni(makaleler):
    return "\n\n".join(f"[{i}] Başlık: {m['baslik']}\n    Özet: {m['ozet']}"
                        for i, m in enumerate(makaleler, 1))

print("retrieval hazir")

In [ ]:
VARYANT = "MET D1228N"          # <-- egitimde OLMAYAN bir varyant dene (or. SMO W535L, IDH2 R172K, EZH2 Y646N)

pmidler, toplam = makale_ara(VARYANT, adet=5)
makaleler = makale_detaylari_al(pmidler)
print(f"{toplam} makale bulundu; {len(makaleler)} tanesi kullanilacak.\n")

# c04_varchat_ollama.py ile BIREBIR ayni system prompt
sistem = (
    f"Sen bir genetik varyant asistanısın. '{VARYANT}' hakkında SADECE aşağıdaki KAYNAKLAR'a "
    "dayanarak yanıt verirsin. Yanıtın DAİMA ve TAMAMEN Türkçe olmalı; başka dil kullanma.\n\n"
    "Kurallar:\n"
    "- Yalnızca kaynaklarda yazan bilgiyi kullan; kendi bilginden ekleme, tahmin etme, uydurma.\n"
    "- Her cümlenin sonuna dayandığı kaynağı yaz: [1], [2]. Kaynağı olmayan cümle yazma.\n"
    "- Cevap kaynaklarda yoksa yalnızca şunu de: 'Bu konuda elimdeki kaynaklarda bilgi yok.'\n\n"
    f"KAYNAKLAR:\n{baglam_metni(makaleler)}"
)

ozet = uret(sistem, f"{VARYANT} varyantını kaynaklı olarak özetle.", max_new_tokens=700)
print("ÖZET:\n")
print(ozet)
print("\nKAYNAKLAR:")
for i, m in enumerate(makaleler, 1):
    print(f"[{i}] {m['baslik']} — https://pubmed.ncbi.nlm.nih.gov/{m['pmid']}/")

## 9) Ollama'ya taşıma — LoRA'yı GGUF'a çevir

Ağır fp16 birleştirme (merge) yapmadan: LoRA adapter'ını **GGUF**'a çevirip yerelde
`ollama create` ile mevcut `qwen2.5:7b` üzerine bindiriyoruz. Adapter küçük (~birkaç yüz MB).

In [ ]:
!git clone -q https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt

# Adapter dizinini GGUF'a cevir (bayrak adlari llama.cpp surumune gore degisebilir)
!python llama.cpp/convert_lora_to_gguf.py {KAYIT_DIZINI} --base Qwen/Qwen2.5-7B-Instruct --outfile varchat-lora.gguf
print("olustu: varchat-lora.gguf")

In [ ]:
# GGUF adapter'ini indir (yerel bilgisayarina)
from google.colab import files
files.download("varchat-lora.gguf")

### Yerelde (kendi bilgisayarında) Ollama'ya bağla

`varchat-lora.gguf`'u indirdikten sonra yanına bir `Modelfile` oluştur:

```
FROM qwen2.5:7b
ADAPTER ./varchat-lora.gguf
```

Sonra terminalde:

```
ollama create varchat -f Modelfile
```

Son olarak `c04_varchat_ollama.py` içinde `MODEL = "qwen2.5:7b"` satırını `MODEL = "varchat"`
yap — artık uygulaman fine-tune edilmiş modeli kullanır.

**Notlar**
- OOM olursa: 4. hücrede model aynı kalır, 5. hücrede `MAX_LEN`'i 3072/2048'e düşür ya da 6. hücrede epoch'u azalt.
- `convert_lora_to_gguf.py` bayrakları llama.cpp sürümüne göre değişebilir; `-h` ile kontrol et.
- Daha büyük model (14B) eğitmek istersen Colab Pro (L4/A100) gerekir; bu notebook'ta yalnızca `MODEL_ID`'yi değiştirmen yeterli.